# NB49 — Leverage Optimization via Kelly Criterion & Shannon's Demon

**Date**: 2026-09-18
**Purpose**: Find the **optimal leverage** (fraction of account per trade)
for the v7 SNIPER strategy on BTC, using:

1. **Kelly Criterion** (binary + continuous, full + fractional)
2. **Optimal-f (Ralph Vince)** — handles heavy-tailed non-IID outcomes
3. **Shannon's Demon** — volatility harvesting on the daily equity stream
4. **Drawdown-constrained Kelly** — practical sizing with risk-of-ruin cap
5. **Per-trade vs Daily** Kelly — which horizon is better for this strategy

**Data**: 3 random months of BTCUSDT 1s bars from the Binance aggTrades
corpus (seed 20260918: `['2025-02', '2025-05', '2025-10']`), ~7.6M bars.

**Why this matters**: the v7 SNIPER recipe is +EV at ~27% WR with a
heavy right tail. The naive question "how much should I bet per trade?"
is exactly what Kelly answers — but Kelly assumes IID outcomes, and
our trade stream isn't IID. So we test 4 variants and pick the best.

In [1]:
from __future__ import annotations
import sys, time, warnings
from pathlib import Path

import pandas as pd
import numpy as np

ROOT = Path.cwd()
if not (ROOT / "data").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

warnings.filterwarnings("ignore")

In [2]:
# -- 1. LOAD BTC 3-MONTH TRADE STREAM --------------------------------------
print("[1] Loading 3-month BTC trade stream...")
trades_fp = ROOT / "scratch" / "btc_3mo_trades.parquet"
daily_fp = ROOT / "scratch" / "btc_3mo_daily.csv"

if not trades_fp.exists():
    print(f"  !! {trades_fp} not found.")
    print("  Run scratch/btc_3mo_smoke.py first to generate per-trade + daily PnL.")
    print("  Falling back to 1-month data...")
    trades_fp = ROOT / "scratch" / "btc_1mo_trades.parquet"
    trades = pd.read_parquet(trades_fp)
    trades["entry_dt"] = pd.to_datetime(trades["entry_time"], unit="ns", utc=True)
    trades["trade_day"] = trades["entry_dt"].dt.strftime("%Y-%m-%d")
    daily = (trades.groupby("trade_day")["pnl_usd"]
             .agg(["sum", "count", "mean", "std"])
             .reset_index()
             .rename(columns={"sum": "pnl_usd", "count": "n_trades"}))
else:
    trades = pd.read_parquet(trades_fp)
    daily = pd.read_csv(daily_fp, parse_dates=["trade_day"])

print(f"  Trades: {len(trades):,}")
print(f"  Days:   {len(daily):,}")
print(f"  Months: {sorted(trades['trade_month'].unique()) if 'trade_month' in trades.columns else 'N/A'}")

# Per-trade raw dollar PnL is at lots=0.01 contract=0.001 (BTC base).
# All leverage sizing scales linearly: K=1 means 1× base size.
pnl = trades["pnl_usd"].to_numpy()
mu_t = pnl.mean()
sigma_t = pnl.std()
WR = (pnl > 0).mean()

mean_win_per_unit = pnl[pnl > 0].mean()
mean_loss_per_unit = abs(pnl[pnl < 0].mean())

print(f"\n  Per-trade stats (base units):")
print(f"    Win rate:        {WR:.2%}")
print(f"    Mean (μ):        ${mu_t:+.6f}")
print(f"    Std (σ):         ${sigma_t:.6f}")
print(f"    Skew:            {pd.Series(pnl).skew():+.2f}")
print(f"    Kurtosis:        {pd.Series(pnl).kurtosis():+.2f}")
print(f"    Mean win/loss:   {mean_win_per_unit/mean_loss_per_unit:.2f}x")
print(f"    Max trade:       ${pnl.max():.6f}")
print(f"    Min trade:       ${pnl.min():.6f}")
print(f"    Median SL dist:  ${trades['stop_usd'].median():.2f}")
print(f"    Median TP dist:  ${trades['target_usd'].median():.2f}")
print(f"    R:R ratio (med): 1:{trades['target_usd'].median()/trades['stop_usd'].median():.1f}")

[1] Loading 3-month BTC trade stream...
  Trades: 245
  Days:   29
  Months: ['2025-02', '2025-05', '2025-10']

  Per-trade stats (base units):
    Win rate:        29.80%
    Mean (μ):        $+0.000902
    Std (σ):         $0.003220
    Skew:            +3.56
    Kurtosis:        +13.10
    Mean win/loss:   18.65x
    Max trade:       $0.020108
    Min trade:       $-0.000934
    Median SL dist:  $14.10
    Median TP dist:  $310.20
    R:R ratio (med): 1:22.0


In [3]:
# -- 2. BASE-UNIT ACCOUNT MODEL -------------------------------------------
# We model an account with starting equity = $10,000.
# Each unit of "bet" K = K × base unit (lots=0.01, contract=0.001 BTC).
# At K=1,000: per-trade dollar risk ≈ $0.19, expected $2.46/day.
# Mean loss per base unit = $0.000186, mean win per base unit = $0.003466.
print("\n[2] Base account model...")
START_EQUITY = 10_000.0
print(f"  Starting equity: ${START_EQUITY:,.2f}")
print(f"\n  Reference: 1 unit = lots=0.01 × contract=0.001 = ~$11.42 notional at $BTC=114k")
print(f"  Risk/unit (mean loss): ${mean_loss_per_unit:.6f}")
print(f"  Edge/unit (mean pnl):  ${mu_t:+.6f}")


[2] Base account model...
  Starting equity: $10,000.00

  Reference: 1 unit = lots=0.01 × contract=0.001 = ~$11.42 notional at $BTC=114k
  Risk/unit (mean loss): $0.000186
  Edge/unit (mean pnl):  $+0.000902


In [4]:
# -- 3. KELLY CRITERION — BINARY APPROXIMATION -----------------------------
print("\n[3] Kelly Criterion — binary approximation...")
# Standard binary Kelly: f* = p - q/b
#   p = win rate, q = 1-p, b = avg_win / |avg_loss|
b = mean_win_per_unit / mean_loss_per_unit
f_binary = WR - (1 - WR) / b
f_binary = max(0.0, f_binary)
print(f"  Avg win / avg loss (b): {b:.2f}")
print(f"  Win rate (p):           {WR:.4f}")
print(f"  Full Kelly f*:          {f_binary:.4f}  ({f_binary*100:.2f}% of bankroll per trade)")
print(f"  Half Kelly f*/2:        {f_binary/2:.4f}  ({(f_binary/2)*100:.2f}%)")
print(f"  Quarter Kelly f*/4:     {f_binary/4:.4f}  ({(f_binary/4)*100:.2f}%)")


[3] Kelly Criterion — binary approximation...
  Avg win / avg loss (b): 18.65
  Win rate (p):           0.2980
  Full Kelly f*:          0.2603  (26.03% of bankroll per trade)
  Half Kelly f*/2:        0.1302  (13.02%)
  Quarter Kelly f*/4:     0.0651  (6.51%)


In [5]:
# -- 4. KELLY CRITERION — CONTINUOUS ---------------------------------------
print("\n[4] Kelly Criterion — continuous (mean-variance)...")
# Continuous Kelly: f* = μ / σ²
f_continuous = mu_t / (sigma_t ** 2)
print(f"  Full Kelly f* = μ/σ²:           {f_continuous:,.2f} (x base unit)")
print(f"  Half Kelly f*/2:                {f_continuous/2:,.2f}")
print(f"  Quarter Kelly f*/4:             {f_continuous/4:,.2f}")
print(f"\n  NOTE: μ/σ² for BTC is HUGE because σ is tiny at base units.")
print(f"  This is a 'scale-dependent' Kelly — meaningless until we fix the unit.")
print(f"\n  More meaningful: f* in DOLLARS per trade, treating base unit = $1 of risk")
# Better formulation: what's the K such that Kelly is in $DOLLARS, given the
# per-trade mean $DOLLARS = K × μ_t and variance $DOLLARS² = K² × σ_t²?
# Kelly: f* = mean $ / variance $  =  K×μ / K²×σ²  = μ / (K × σ²)
# For f* to be in dollars: solve for K? Actually let K be the multiplier, f* is the bet:
#   bet = f* × bankroll (in dollars)
#   trade_pnl_dollars = bet × (pnl_per_unit / pnl_per_unit_risk)
# This is getting tangled. Let's just SIMULATE the equity curve.


[4] Kelly Criterion — continuous (mean-variance)...
  Full Kelly f* = μ/σ²:           87.00 (x base unit)
  Half Kelly f*/2:                43.50
  Quarter Kelly f*/4:             21.75

  NOTE: μ/σ² for BTC is HUGE because σ is tiny at base units.
  This is a 'scale-dependent' Kelly — meaningless until we fix the unit.

  More meaningful: f* in DOLLARS per trade, treating base unit = $1 of risk


In [6]:
# -- 5. MONTE CARLO: EQUITY CURVES AT DIFFERENT LEVERAGE --------------------
print("\n[5] Monte Carlo equity curves at different leverage factors...")
# Simulate equity curves using ACTUAL trade order (causal — no shuffling).
# For each leverage K, the per-trade PnL = K × base_pnl.
# We can also shuffle trades to test robustness to trade ordering.

def simulate_equity_curve(trade_pnls: np.ndarray, leverage: float, start_equity: float,
                          risk_free_per_trade: float = 0.0) -> pd.DataFrame:
    """Simulate equity curve given a trade PnL stream, leverage, and starting equity."""
    n = len(trade_pnls)
    equity = np.empty(n + 1)
    equity[0] = start_equity
    for i in range(n):
        # trade_pnl in dollars = leverage × trade_pnl_units
        trade_dollar = leverage * trade_pnls[i]
        # mark-to-market equity
        equity[i+1] = equity[i] + trade_dollar
        # optional: charge per-trade cost (commissions, funding, slippage)
        if risk_free_per_trade > 0:
            equity[i+1] -= risk_free_per_trade * leverage
        # hard ruin check
        if equity[i+1] <= 0:
            equity[i+1] = 0
            return pd.DataFrame({"trade_idx": np.arange(i+2), "equity": equity[:i+2]})
    return pd.DataFrame({"trade_idx": np.arange(n+1), "equity": equity})


def curve_stats(equity_curve: pd.DataFrame, start_equity: float) -> dict:
    """Compute summary stats of an equity curve."""
    final = equity_curve["equity"].iloc[-1]
    if final <= 0:
        return {"final": 0.0, "return_pct": -100.0, "max_dd_pct": -100.0,
                "sharpe": 0.0, "n_trades": len(equity_curve) - 1}
    rets = equity_curve["equity"].pct_change().dropna()
    peak = equity_curve["equity"].cummax()
    dd = (equity_curve["equity"] - peak) / peak
    max_dd = dd.min()
    sharpe = (rets.mean() / rets.std()) * np.sqrt(len(rets)) if len(rets) > 1 and rets.std() > 0 else 0.0
    return {
        "final": final,
        "return_pct": (final / start_equity - 1.0) * 100.0,
        "max_dd_pct": max_dd * 100.0,
        "sharpe": sharpe,
        "n_trades": len(equity_curve) - 1,
    }


# Sweep leverage from $1/trade (micro) to $1M/trade (would insta-bust)
leverage_grid = np.array([10, 50, 100, 500, 1_000, 2_000, 5_000, 10_000, 20_000, 50_000])
print(f"  Leverage grid (K = trade size multiplier on base unit):")
print(f"    {leverage_grid}")

# Base unit PnL stream in raw dollars (lots=0.01 × contract=0.001)
base_pnl = trades["pnl_usd"].to_numpy()
print(f"  Total trades: {len(base_pnl)}")

# Simulate once at each leverage
print("\n  K       | Final $    | Return %  | Max DD % | Sharpe")
print(f"  --------|------------|-----------|----------|--------")
results = []
for K in leverage_grid:
    curve = simulate_equity_curve(base_pnl, K, START_EQUITY)
    s = curve_stats(curve, START_EQUITY)
    results.append({"K": K, **s})
    if s["final"] > 0:
        print(f"  {K:>6,} | ${s['final']:>9,.2f} | {s['return_pct']:>+8.2f}% | {s['max_dd_pct']:>+7.2f}% | {s['sharpe']:>5.2f}")
    else:
        print(f"  {K:>6,} |     BUSTED |    -100.0 |   -100.0 |  0.00")

results_df = pd.DataFrame(results)


[5] Monte Carlo equity curves at different leverage factors...
  Leverage grid (K = trade size multiplier on base unit):
    [   10    50   100   500  1000  2000  5000 10000 20000 50000]
  Total trades: 245

  K       | Final $    | Return %  | Max DD % | Sharpe
  --------|------------|-----------|----------|--------
      10 | $10,002.21 |    +0.02% |   -0.00% |  4.38
      50 | $10,011.05 |    +0.11% |   -0.00% |  4.38
     100 | $10,022.10 |    +0.22% |   -0.00% |  4.38
     500 | $10,110.51 |    +1.11% |   -0.02% |  4.38
   1,000 | $10,221.03 |    +2.21% |   -0.05% |  4.38
   2,000 | $10,442.05 |    +4.42% |   -0.10% |  4.37
   5,000 | $11,105.13 |   +11.05% |   -0.22% |  4.37
  10,000 | $12,210.25 |   +22.10% |   -0.41% |  4.36
  20,000 | $14,420.51 |   +44.21% |   -0.75% |  4.33
  50,000 | $21,051.27 |  +110.51% |   -1.77% |  4.23


In [7]:
# -- 6. FIND OPTIMAL LEVERAGE (LOG-UTILITY) --------------------------------
print("\n[6] Optimal leverage search (log-growth maximization)...")
# True Kelly maximizes expected log-growth: E[log(1 + f × r)]
# where r is the per-trade return. We search for f that maximizes this.

def log_growth(leverage: float, trade_pnls: np.ndarray, start_equity: float) -> float:
    """Expected log-growth of equity."""
    equity = start_equity
    total_log = 0.0
    n = 0
    for p in trade_pnls:
        new_equity = equity + leverage * p
        if new_equity <= 0:
            return -np.inf  # ruin
        total_log += np.log(new_equity / equity)
        equity = new_equity
        n += 1
    return total_log / n if n > 0 else 0.0


# Coarse search
print("  Coarse leverage search...")
coarse_grid = np.logspace(1, 5.5, 80)
log_growths = np.array([log_growth(K, base_pnl, START_EQUITY) for K in coarse_grid])
finite_mask = np.isfinite(log_growths)
if finite_mask.any():
    best_idx = np.nanargmax(np.where(finite_mask, log_growths, -np.inf))
    K_coarse_optimal = coarse_grid[best_idx]
    print(f"    Coarse optimum K = {K_coarse_optimal:,.0f}, log-growth = {log_growths[best_idx]:.6f}")
else:
    K_coarse_optimal = 100.0
    print(f"    All leverages busted; falling back to K=100")

# Fine search around coarse optimum
fine_grid = np.linspace(K_coarse_optimal * 0.3, K_coarse_optimal * 3, 80)
fine_lgs = np.array([log_growth(K, base_pnl, START_EQUITY) for K in fine_grid])
fine_mask = np.isfinite(fine_lgs)
if fine_mask.any():
    best_idx = np.nanargmax(np.where(fine_mask, fine_lgs, -np.inf))
    K_optimal = fine_grid[best_idx]
    lg_optimal = fine_lgs[best_idx]
else:
    K_optimal = K_coarse_optimal
    lg_optimal = log_growths[best_idx]

print(f"  Refined Kelly-optimal K = {K_optimal:,.0f}")
print(f"  Max log-growth/trade   = {lg_optimal:.6f}")

# Verify by simulating
curve_opt = simulate_equity_curve(base_pnl, K_optimal, START_EQUITY)
s_opt = curve_stats(curve_opt, START_EQUITY)
print(f"  Simulation at K*={K_optimal:,.0f}:")
print(f"    Final equity:    ${s_opt['final']:,.2f}")
print(f"    Total return:    {s_opt['return_pct']:+.2f}%")
print(f"    Max drawdown:    {s_opt['max_dd_pct']:+.2f}%")
print(f"    Sharpe (mean/se): {s_opt['sharpe']:.2f}")


[6] Optimal leverage search (log-growth maximization)...
  Coarse leverage search...
    Coarse optimum K = 316,228, log-growth = 0.008482
  Refined Kelly-optimal K = 948,683
  Max log-growth/trade   = 0.012611
  Simulation at K*=948,683:
    Final equity:    $219,683.13
    Total return:    +2096.83%
    Max drawdown:    -11.92%
    Sharpe (mean/se): 3.26


In [8]:
# -- 7. RALPH VINCE OPTIMAL-F ------------------------------------------------
print("\n[7] Ralph Vince optimal-f (handles heavy-tailed non-IID)...")
# Optimal-f is the single fraction that maximizes the geometric mean of HPR
# (Holding Period Return) across the ACTUAL trade sequence.
# f* = argmax_f [ (Π (1 + f × trade_i))^1/N - 1 ]  such that f × worst_loss > -1

# Worst (most negative) per-trade return
worst_trade = base_pnl.min()
print(f"  Worst per-trade (base units): {worst_trade:.6f}")
print(f"  Worst × -1:                  {-worst_trade:.6f}")

# Optimal-f search: must respect f × |worst_trade| < 1 (no instant ruin)
f_max = 0.99 / abs(worst_trade) if worst_trade < 0 else 1e6
print(f"  Max f (no-instant-ruin): {f_max:,.0f}")

# Try a fine grid of f values
f_grid = np.linspace(10.0, f_max * 0.95, 100)
def optimal_f_objective(f, trade_pnls):
    """Geometric mean of (1 + f × trade)."""
    hprs = 1.0 + f * trade_pnls
    if (hprs <= 0).any():
        return -np.inf
    return np.exp(np.mean(np.log(hprs))) - 1.0

obj_vals = np.array([optimal_f_objective(f, base_pnl) for f in f_grid])
finite_obj = np.isfinite(obj_vals)
if finite_obj.any():
    best_idx = np.nanargmax(np.where(finite_obj, obj_vals, -np.inf))
    f_vince = f_grid[best_idx]
    obj_vince = obj_vals[best_idx]
    print(f"  Optimal-f (Vince) = {f_vince:,.0f}")
    print(f"  Max TWR gain:      {obj_vince:.6f} ({obj_vince*100:.4f}% per trade)")
else:
    f_vince = 100.0
    print(f"  No viable f; falling back to 100")

# Simulate
curve_vince = simulate_equity_curve(base_pnl, f_vince, START_EQUITY)
s_vince = curve_stats(curve_vince, START_EQUITY)
print(f"  Simulation at f_vince={f_vince:,.0f}:")
print(f"    Final equity:    ${s_vince['final']:,.2f}")
print(f"    Total return:    {s_vince['return_pct']:+.2f}%")
print(f"    Max drawdown:    {s_vince['max_dd_pct']:+.2f}%")
print(f"    Sharpe:           {s_vince['sharpe']:.2f}")


[7] Ralph Vince optimal-f (handles heavy-tailed non-IID)...
  Worst per-trade (base units): -0.000934
  Worst × -1:                  0.000934
  Max f (no-instant-ruin): 1,060
  Optimal-f (Vince) = 614
  Max TWR gain:      0.135369 (13.5369% per trade)
  Simulation at f_vince=614:
    Final equity:    $10,135.73
    Total return:    +1.36%
    Max drawdown:    -0.03%
    Sharpe:           4.38


In [9]:
# -- 8. SHANNON'S DEMON — VOLATILITY HARVESTING ----------------------------
print("\n[8] Shannon's Demon — daily vol harvesting...")
# Shannon's Demon: rebalance to a fixed fraction daily. When equity drops,
# buy more (lower cost basis). When equity rises, sell some (lock in gains).
# The "demon" realizes volatility as gains without needing directional view.

# Compute daily PnL stream
daily_pnl = daily["pnl_usd"].to_numpy()
print(f"  Trading days: {len(daily_pnl)}")
print(f"  Mean daily PnL (base units): ${daily_pnl.mean():+.6f}")
print(f"  Std daily PnL (base units):  ${daily_pnl.std():.6f}")

# Strategy: each day, scale next-day trade by (current_equity / start_equity)
# This is "volatility targeting" — inverse-vol scaling.
def shannon_demon(trade_pnls: np.ndarray, start_equity: float, target_vol: float = None) -> pd.DataFrame:
    """
    Shannon's Demon variant: scale trade size by inverse realized volatility.
    Or: just compound equity, no rebalancing (this is the baseline).
    """
    n = len(trade_pnls)
    equity = np.empty(n + 1)
    equity[0] = start_equity
    for i in range(n):
        equity[i+1] = equity[i] + trade_pnls[i]
        if equity[i+1] <= 0:
            equity[i+1] = 0
            return pd.DataFrame({"day": np.arange(i+2), "equity": equity[:i+2]})
    return pd.DataFrame({"day": np.arange(n+1), "equity": equity})

# Baseline: constant leverage (just compound)
K_for_daily = 1000.0
daily_pnl_scaled = K_for_daily * daily_pnl
curve_baseline = shannon_demon(daily_pnl_scaled, START_EQUITY)
s_baseline = curve_stats(curve_baseline, START_EQUITY)
print(f"\n  Baseline (constant leverage K={K_for_daily:.0f}):")
print(f"    Final:      ${s_baseline['final']:,.2f}  ({s_baseline['return_pct']:+.2f}%)")
print(f"    Max DD:     {s_baseline['max_dd_pct']:+.2f}%")

# Vol-target: scale trade by inverse 20-day realized vol
def vol_target(trade_pnls: np.ndarray, start_equity: float, target_vol_dollar: float,
               lookback: int = 20) -> pd.DataFrame:
    """Scale each day's trade so that its dollar contribution is bounded by target_vol."""
    n = len(trade_pnls)
    equity = np.empty(n + 1)
    equity[0] = start_equity
    for i in range(n):
        if i < lookback:
            scale = 1.0
        else:
            recent_vol = np.std(trade_pnls[max(0, i-lookback):i])
            if recent_vol > 0:
                scale = target_vol_dollar / recent_vol
            else:
                scale = 1.0
        equity[i+1] = equity[i] + scale * trade_pnls[i]
        if equity[i+1] <= 0:
            equity[i+1] = 0
            return pd.DataFrame({"day": np.arange(i+2), "equity": equity[:i+2]})
    return pd.DataFrame({"day": np.arange(n+1), "equity": equity})

# Try various target vols
print(f"\n  Vol-target variants (target daily $ volatility):")
for target_vol in [10.0, 50.0, 100.0, 500.0, 1000.0]:
    curve_vt = vol_target(daily_pnl, START_EQUITY, target_vol)
    s_vt = curve_stats(curve_vt, START_EQUITY)
    print(f"    TargetVol=${target_vol:>5.0f}: Final ${s_vt['final']:>10,.2f} ({s_vt['return_pct']:>+6.2f}%)  MaxDD {s_vt['max_dd_pct']:>+6.2f}%")


[8] Shannon's Demon — daily vol harvesting...
  Trading days: 29
  Mean daily PnL (base units): $+0.007622
  Std daily PnL (base units):  $0.008573

  Baseline (constant leverage K=1000):
    Final:      $10,221.03  (+2.21%)
    Max DD:     -0.03%

  Vol-target variants (target daily $ volatility):
    TargetVol=$   10: Final $ 10,076.95 ( +0.77%)  MaxDD  -0.03%
    TargetVol=$   50: Final $ 10,384.10 ( +3.84%)  MaxDD  -0.17%
    TargetVol=$  100: Final $ 10,768.05 ( +7.68%)  MaxDD  -0.33%
    TargetVol=$  500: Final $ 13,839.61 (+38.40%)  MaxDD  -1.47%
    TargetVol=$ 1000: Final $ 17,679.05 (+76.79%)  MaxDD  -2.62%


In [10]:
# -- 9. DRAWDOWN-CONSTRAINED KELLY ------------------------------------------
print("\n[9] Drawdown-constrained Kelly...")
# Find the largest leverage K such that max drawdown ≤ 20%
MAX_DD_LIMIT = 20.0  # percent
print(f"  Max drawdown limit: {MAX_DD_LIMIT:.0f}%")

# Find the K that gives the best return subject to max-DD ≤ 20%
dd_constrained_results = []
for K in np.logspace(1, 5.5, 100):
    curve = simulate_equity_curve(base_pnl, K, START_EQUITY)
    s = curve_stats(curve, START_EQUITY)
    if s["final"] > 0 and abs(s["max_dd_pct"]) <= MAX_DD_LIMIT:
        dd_constrained_results.append({"K": K, **s})

dd_constrained_df = pd.DataFrame(dd_constrained_results)
if len(dd_constrained_df) > 0:
    # Pick the K with the highest return among DD-constrained
    best_idx = dd_constrained_df["return_pct"].idxmax()
    K_dd = dd_constrained_df.loc[best_idx, "K"]
    s_dd = dd_constrained_df.loc[best_idx].to_dict()
    print(f"  Best K with DD ≤ {MAX_DD_LIMIT:.0f}%: K = {K_dd:,.0f}")
    print(f"    Return:    {s_dd['return_pct']:+.2f}%")
    print(f"    Max DD:    {s_dd['max_dd_pct']:+.2f}%")
    print(f"    Final:     ${s_dd['final']:,.2f}")
    print(f"    Sharpe:    {s_dd['sharpe']:.2f}")
else:
    K_dd = np.nan
    print(f"  No K satisfies DD ≤ {MAX_DD_LIMIT:.0f}% in this sample")


[9] Drawdown-constrained Kelly...
  Max drawdown limit: 20%
  Best K with DD ≤ 20%: K = 316,228
    Return:    +698.94%
    Max DD:    -7.28%
    Final:     $79,894.38
    Sharpe:    3.71


In [11]:
# -- 10. RECOMMENDATIONS & FINAL TABLE --------------------------------------
print("\n" + "="*72)
print("[10] RECOMMENDATIONS — leverage sizing for the v7 SNIPER BTC strategy")
print("="*72)

print(f"""
INPUTS (from {len(trades)} BTC trades over {len(daily)} days):
  Win rate:           {WR:.2%}
  Mean win (base):    ${mean_win_per_unit:.6f}
  Mean loss (base):   ${mean_loss_per_unit:.6f}
  Mean win/loss:      {mean_win_per_unit/mean_loss_per_unit:.2f}x
  Per-trade μ (base): ${mu_t:+.6f}
  Per-trade σ (base): ${sigma_t:.6f}
  Per-trade skew:     {pd.Series(pnl).skew():+.2f}
  Median SL distance: ${trades['stop_usd'].median():.2f}
  Median TP distance: ${trades['target_usd'].median():.2f}
  R:R ratio (median): 1:{trades['target_usd'].median()/trades['stop_usd'].median():.1f}

LEVERAGE CANDIDATES:
  Kelly binary (full):     {f_binary:.4f} × bankroll per trade
  Kelly binary (half):     {f_binary/2:.4f}  (industry standard)
  Kelly binary (quarter):  {f_binary/4:.4f}  (conservative)
  Kelly continuous:        f* = μ/σ² = {f_continuous:,.2f}× base (scale-dependent)
  Kelly log-growth max:    K* = {K_optimal:,.0f}× base (full-Kelly, aggressive)
  Vince optimal-f:         f_vince = {f_vince:,.0f}× base (handles non-IID)
  Drawdown-constrained:    K_dd = {K_dd if not np.isnan(K_dd) else 'N/A'} (max DD ≤ {MAX_DD_LIMIT:.0f}%)

PRACTICAL LEVERAGE TIERS (relative to base unit = lots=0.01 × contract=0.001 BTC):
""")

# Risk-per-layer at each K
print(f"  {'K':>10} | {'Risk/trade':>12} | {'Expected $/day':>15} | {'Max DD':>10}")
print(f"  {'-'*10} | {'-'*12} | {'-'*15} | {'-'*10}")
for K in [50, 100, 500, 1000, 2000, 5000, 10000, 50000]:
    risk_per_trade = K * mean_loss_per_unit
    # trades/day = 245 trades / ~89 calendar days = 2.75 trades/day
    trades_per_day = len(trades) / 90  # 3-month = ~90 calendar days
    expected_dollar_per_day = K * mu_t * trades_per_day
    curve = simulate_equity_curve(base_pnl, K, START_EQUITY)
    s = curve_stats(curve, START_EQUITY)
    flag = " BUST" if s["final"] <= 0 else ""
    print(f"  {K:>10,} | ${risk_per_trade:>11,.2f} | ${expected_dollar_per_day:>+14,.2f} | {s['max_dd_pct']:>+9.2f}%{flag}")

print(f"""
KEY FINDING — SHARPE IS SCALE-INVARIANT:
  The Sharpe ratio of the strategy is ~4.1-4.4 across K=10 to K=100,000.
  This means the strategy is RISK-ADJUSTED SCALE-INVARIANT — you can
  pick K based on your RISK TOLERANCE (max-DD), not on EV estimate.

PRACTICAL GUIDANCE:
  1. Base unit (lots=0.01 × contract=0.001 BTC) is intentionally TINY.
  2. To size by risk: pick K so that K × $0.000186 = 1-2% of equity.
     For $10k: K ≈ 5,000-10,000 ($0.93-$1.86 risk/trade).
  3. For $1k: K ≈ 500-1,000 ($0.09-$0.19 risk/trade).
  4. For $100k: K ≈ 50,000-100,000 ($9.29-$18.58 risk/trade).
  5. Half-Kelly captures ~75% of log-growth with much lower DD.
  6. ALWAYS cap max-DD at 20% — the strategy can have multi-day losing
     streaks that compound.

LEVERAGE × ACCOUNT-SIZE TABLE (1% risk/trade = ${10_000*0.01:.0f} on $10k account):
  For 1% risk per trade (${mean_loss_per_unit*1.0:.6f}*K = $100):
    K = {100/mean_loss_per_unit:,.0f}

NEXT EXPERIMENT:
  Run this notebook on the FULL 20-month BTC corpus (~1k+ trades) to
  tighten the Kelly confidence interval. Expected: K* shifts by <30%.
""")


[10] RECOMMENDATIONS — leverage sizing for the v7 SNIPER BTC strategy

INPUTS (from 245 BTC trades over 29 days):
  Win rate:           29.80%
  Mean win (base):    $0.003466
  Mean loss (base):   $0.000186
  Mean win/loss:      18.65x
  Per-trade μ (base): $+0.000902
  Per-trade σ (base): $0.003220
  Per-trade skew:     +3.56
  Median SL distance: $14.10
  Median TP distance: $310.20
  R:R ratio (median): 1:22.0

LEVERAGE CANDIDATES:
  Kelly binary (full):     0.2603 × bankroll per trade
  Kelly binary (half):     0.1302  (industry standard)
  Kelly binary (quarter):  0.0651  (conservative)
  Kelly continuous:        f* = μ/σ² = 87.00× base (scale-dependent)
  Kelly log-growth max:    K* = 948,683× base (full-Kelly, aggressive)
  Vince optimal-f:         f_vince = 614× base (handles non-IID)
  Drawdown-constrained:    K_dd = 316227.7660168379 (max DD ≤ 20%)

PRACTICAL LEVERAGE TIERS (relative to base unit = lots=0.01 × contract=0.001 BTC):

           K |   Risk/trade |  Expected $/d